# Claude 멀티 펑션콜링 (Parallel Tool Use)

Claude는 한 번의 응답에서 **여러 개의 `tool_use` 블록**을 동시에 반환할 수 있습니다 (병렬 펑션콜링, 기본 활성화).

핵심 규칙:
- 응답의 `stop_reason == "tool_use"` 이면 도구 호출 요청
- 여러 도구를 실행한 뒤 **모든 `tool_result`를 하나의 user 메시지로 묶어서** 반환해야 함 (나눠 보내면 모델이 병렬 호출을 점점 안 하게 됨)
- `end_turn`이 나올 때까지 루프 반복

In [1]:
import json

import anthropic
from dotenv import load_dotenv

load_dotenv()  # .env 의 ANTHROPIC_API_KEY 로드

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

## 1. 도구 정의

`description`에는 도구가 무엇을 하는지뿐 아니라 **언제 호출해야 하는지**를 명시하는 것이 좋습니다.

In [2]:
TOOLS = [
    {
        "name": "get_weather",
        "description": "지정한 도시의 현재 날씨와 기온을 조회한다. 사용자가 날씨나 기온을 물으면 이 도구를 호출한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "도시 이름 (영문), 예: Seoul, Tokyo, Paris"}
            },
            "required": ["city"],
        },
    },
    {
        "name": "get_exchange_rate",
        "description": "USD 기준 환율을 조회한다. 사용자가 환율이나 통화 변환을 물으면 이 도구를 호출한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "currency": {"type": "string", "description": "통화 코드, 예: KRW, JPY, EUR"}
            },
            "required": ["currency"],
        },
    },
]

## 2. 도구 구현 (모의 데이터)

실제로는 외부 API를 호출하겠지만, 여기서는 고정된 모의 데이터를 반환합니다.

In [3]:
MOCK_WEATHER = {
    "Seoul": {"condition": "맑음", "temp_c": 31},
    "Tokyo": {"condition": "흐림", "temp_c": 29},
    "Paris": {"condition": "비", "temp_c": 22},
}
MOCK_RATES = {"KRW": 1385.2, "JPY": 157.8, "EUR": 0.92}


def get_weather(city: str) -> dict:
    if city not in MOCK_WEATHER:
        raise ValueError(f"'{city}' 날씨 정보 없음. 가능한 도시: {list(MOCK_WEATHER)}")
    return {"city": city, **MOCK_WEATHER[city]}


def get_exchange_rate(currency: str) -> dict:
    if currency not in MOCK_RATES:
        raise ValueError(f"'{currency}' 환율 정보 없음. 가능한 통화: {list(MOCK_RATES)}")
    return {"base": "USD", "currency": currency, "rate": MOCK_RATES[currency]}


TOOL_FUNCTIONS = {"get_weather": get_weather, "get_exchange_rate": get_exchange_rate}

## 3. 에이전트 루프

한 응답에 담긴 여러 `tool_use` 블록을 전부 실행하고, 결과를 **하나의 user 메시지**로 묶어 반환합니다.
실패한 도구는 결과를 빼먹지 말고 `is_error: True`로 돌려줘야 모델이 스스로 복구할 수 있습니다.

In [4]:
def run_agent(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=16000,
            thinking={"type": "adaptive"},
            tools=TOOLS,
            messages=messages,
        )
        print(f"[stop_reason] {response.stop_reason}")

        if response.stop_reason != "tool_use":
            break

        # thinking / tool_use 블록을 포함한 응답 전체를 그대로 히스토리에 추가
        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            print(f"  → {block.name}({json.dumps(block.input, ensure_ascii=False)})")
            try:
                result = TOOL_FUNCTIONS[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result, ensure_ascii=False),
                })
            except Exception as e:
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": f"Error: {e}",
                    "is_error": True,
                })

        # 병렬 호출 결과는 반드시 하나의 user 메시지로 묶어 반환
        messages.append({"role": "user", "content": tool_results})

    return next((b.text for b in response.content if b.type == "text"), "")

## 4. 실행

서로 독립적인 조회 6건(날씨 3 + 환율 3)이므로, Claude가 한 턴에 여러 `tool_use`를 병렬로 요청하는 것을 볼 수 있습니다.

In [5]:
answer = run_agent(
    "서울, 도쿄, 파리의 현재 날씨를 알려주고, 1 USD가 원화·엔화·유로로 각각 얼마인지도 알려줘."
)
print("\n=== 최종 답변 ===")
print(answer)

[stop_reason] tool_use
  → get_weather({"city": "Seoul"})
  → get_weather({"city": "Tokyo"})
  → get_weather({"city": "Paris"})
  → get_exchange_rate({"currency": "KRW"})
  → get_exchange_rate({"currency": "JPY"})
  → get_exchange_rate({"currency": "EUR"})
[stop_reason] end_turn

=== 최종 답변 ===
요청하신 정보를 정리해 드립니다.

## 🌤️ 현재 날씨
| 도시 | 날씨 | 기온 |
|------|------|------|
| 서울 | 맑음 ☀️ | 31°C |
| 도쿄 | 흐림 ☁️ | 29°C |
| 파리 | 비 🌧️ | 22°C |

## 💱 환율 (1 USD 기준)
- **원화(KRW):** 1,385.2원
- **엔화(JPY):** 157.8엔
- **유로(EUR):** 0.92유로

서울과 도쿄는 덥고, 파리는 비가 내리며 선선한 편이네요. 파리에 가신다면 우산을 챙기시는 걸 추천드립니다! ☂️


## 참고

- **Tool Runner**: 위 루프를 SDK가 대신 돌려주는 베타 헬퍼도 있습니다 — `@anthropic.beta_tool` 데코레이터 + `client.beta.messages.tool_runner(...)`. 함수 시그니처에서 스키마를 자동 생성해줘서 프로덕션에서는 이쪽이 권장됩니다.
- **강제 호출**: `tool_choice={"type": "tool", "name": "get_weather"}`로 특정 도구를 강제할 수 있고, `disable_parallel_tool_use: True`로 병렬 호출을 끌 수도 있습니다.
- **strict 모드**: 도구 정의에 `"strict": True` + `additionalProperties: false`를 넣으면 `tool_use.input`이 스키마를 정확히 따르는 것이 보장됩니다.